# v4.1 — Replace NKD=F with ^N225

| | |
|---|---|
| **Version** | v4.1 |
| **Key change** | SGX proxy: `NKD=F` (CME Nikkei USD futures) → `^N225` (Nikkei 225 cash index, JPY) |
| **Data source** | 2024 real 1-min NSE option CSVs |
| **Lookahead** | None — D-1 close used |
| **Baseline** | v2/backtest |

**Why**: NKD=F adds USD/JPY currency noise. ^N225 is the actual Nikkei 225 index
that SGX/GIFT Nifty futures track. GIFT Nifty itself has no yfinance daily series.

**One-line change in Cell 3**: `'SGX': 'NKD=F'` → `'SGX': '^N225'`


In [26]:
VERSION    = 'v4.1'
KEY_CHANGE = 'SGX proxy: NKD=F → ^N225 (Nikkei 225 cash, JPY-denominated)'
DATA_SRC   = 'real_2024_options'

SL_PCT    = 0.15
TP_PCT    = 0.40
BASE_LOTS = 2
MAX_LOTS  = 10
DTE0_LOTS = 5
LOT_SIZE  = 75
BEAR_N    = 10
BASE_RATE = 54.5
ENTRY_TIME      = '09:25'
EXIT_TIME       = '11:15'
INITIAL_CAPITAL = 3000.0   # fixed starting capital for all versions (Rs)

print(f"Breakeven win rate: {SL_PCT/(SL_PCT+TP_PCT)*100:.1f}%")


Breakeven win rate: 27.3%


In [27]:
import sys, os
from pathlib import Path

HERE    = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
V4_DIR  = HERE.parent if HERE.name.startswith('v4') else HERE.parent.parent / 'v4'
sys.path.insert(0, str(V4_DIR))

from _engine import (simulate_trades, compute_metrics, save_results,
                     sig_map, combo_fires, load_reliable_signals, print_summary)

import pandas as pd
import numpy  as np
import yfinance as yf
import pickle, warnings
from datetime import date, timedelta, time as dtime
from pathlib import Path
warnings.filterwarnings('ignore')


In [28]:
# ── Paths ──────────────────────────────────────────────────────────────────────
# HERE.parent.parent == gap_trading/ for both v2/backtest and v4/v4.x layouts
GAP_TRADING     = HERE.parent.parent
DATA_2024       = GAP_TRADING / 'v2' / 'backtesting_2024_options' / '2024'
SIGNALS_CSV     = GAP_TRADING / 'v2' / 'v2_reliable_signals.csv'
NIFTY_SPOT_DIR  = DATA_2024 / '2024Nifty'
EXPIRY_CSV      = DATA_2024 / 'expiry.csv'
CACHE_FILE      = HERE / 'results' / f'trade_cache_{VERSION.replace(".", "_")}.pkl'
# ── NSE holidays and event days (2024) ────────────────────────────────────────
NSE_HOLIDAYS = {
    date(2024,  1, 22), date(2024,  3, 25), date(2024,  3, 29),
    date(2024,  4, 14), date(2024,  4, 17), date(2024,  5,  1),
    date(2024,  6, 17), date(2024,  7, 17), date(2024,  8, 15),
    date(2024, 10,  2), date(2024, 11, 15), date(2024, 12, 25),
}
EVENT_DAYS = {
    date(2024,  2,  1),   # Union Budget
    date(2024,  2,  8),   # RBI MPC
    date(2024,  4,  5),   # RBI MPC
    date(2024,  6,  7),   # RBI MPC
    date(2024,  8,  8),   # RBI MPC
    date(2024, 10,  9),   # RBI MPC
    date(2024, 12,  6),   # RBI MPC
}
# ── Expiry helpers ─────────────────────────────────────────────────────────────
expiry_df = pd.read_csv(EXPIRY_CSV)
expiry_df.columns = [c.strip() for c in expiry_df.columns]
# Column is 'ExpiryDate' in DDMMMYY format (e.g. '04JAN24')
_ecol = [c for c in expiry_df.columns if 'expiry' in c.lower() or 'date' in c.lower()][0]
expiry_dates = sorted(
    pd.to_datetime(expiry_df[_ecol].str.strip(), format='%d%b%y').dt.date.tolist()
)

def nearest_expiry(d):
    for e in expiry_dates:
        if e >= d:
            return e
    return expiry_dates[-1]

def expiry_folder(exp):
    return '2024' + exp.strftime('%b').capitalize()
# ── Global market data (yfinance, daily) ──────────────────────────────────────
print("Fetching global market data ...", end=' ', flush=True)
START = '2023-12-15'
END   = '2025-01-10'
TICKERS = {'SP500': '^GSPC', 'SGX': '^N225', 'DAX': '^GDAXI',
           'VIX': '^VIX', 'NIFTY': '^NSEI'}

raw = {}
for name, ticker in TICKERS.items():
    try:
        df = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if df.index.tz is None:
            df.index = df.index.tz_localize('UTC')
        raw[name] = df[['Close']].rename(columns={'Close': name})
    except Exception as e:
        print(f"  [warn] {name}: {e}")
        raw[name] = pd.DataFrame()

gdf = pd.concat([v for v in raw.values() if not v.empty], axis=1)
gdf.index = gdf.index.date
gdf = gdf.ffill()
print(f"done. {len(gdf)} rows.")
# ── Load signal combos ────────────────────────────────────────────────────────
bear_combos = load_reliable_signals(SIGNALS_CSV, bear_n=BEAR_N, base_rate=BASE_RATE)
print(f"Loaded {len(bear_combos)} bearish combos from {SIGNALS_CSV.name}")
for i, c in enumerate(bear_combos, 1):
    print(f"  {i:>2}. {c}")
# ── NIFTY spot 1-min data ─────────────────────────────────────────────────────
spot_cache = {}
for f in sorted(NIFTY_SPOT_DIR.glob('Nifty-2024*.csv')):
    try:
        s = pd.read_csv(f)
        s.columns = [c.strip().lower() for c in s.columns]
        s['datetime'] = pd.to_datetime(s['datetime'])
        s['date'] = s['datetime'].dt.date
        s['time'] = s['datetime'].dt.time
        for d, grp in s.groupby('date'):
            spot_cache[d] = grp.reset_index(drop=True)
    except Exception as e:
        print(f"  [warn] spot {f.name}: {e}")
print(f"Spot data loaded for {len(spot_cache)} days.")
# ── Build signal_days dict ────────────────────────────────────────────────────
if CACHE_FILE.exists():
    with open(CACHE_FILE, 'rb') as f:
        signal_days = pickle.load(f)
    print(f"Loaded trade cache: {len(signal_days)} dates, "
          f"{sum(1 for v in signal_days.values() if v is not None)} with signal.")
else:
    signal_days = {}
    skipped = []
    LOT_SIZE_LOCAL = 75
    STRIKE_STEP    = 50

    all_dates = sorted([d for d in gdf.index if isinstance(d, date)
                        and date(2024,1,1) <= d <= date(2024,12,31)])

    for d in all_dates:
        # ── Skip conditions ────────────────────────────────────────────────
        if d.weekday() == 0:           # Monday
            skipped.append((d, 'Monday')); continue
        if d in NSE_HOLIDAYS:
            skipped.append((d, 'Holiday')); continue
        if d in EVENT_DAYS:
            skipped.append((d, 'EventDay')); continue

        # ── Global data row ────────────────────────────────────────────────
        prev_rows = [x for x in gdf.index if x < d]
        if len(prev_rows) < 2:
            skipped.append((d, 'NoGlobalData')); continue
        prev  = prev_rows[-1]
        prev2 = prev_rows[-2]

        def _ret(col, d1, d2):
            try:
                return float((gdf.loc[d1, col] - gdf.loc[d2, col]) / gdf.loc[d2, col])
            except Exception:
                return None

        sp500_ret = _ret('SP500', prev, prev2)
        sgx_ret   = _ret('SGX',   prev, prev2)
        dax_ret   = _ret('DAX',   prev, prev2)
        vix_ret   = _ret('VIX',   prev, prev2)
        pind_ret  = _ret('NIFTY', prev, prev2)

        # ── NIFTY open (9:15) for gap ──────────────────────────────────────
        spot = spot_cache.get(d)
        if spot is None:
            skipped.append((d, 'NoSpotData')); continue
        open_row = spot[spot['time'] == dtime(9, 15)]
        if open_row.empty:
            skipped.append((d, 'No915Open')); continue
        nifty_open = float(open_row.iloc[0]['open'])

        nifty_prev_close = float(gdf.loc[prev, 'NIFTY']) if prev in gdf.index else None
        if nifty_prev_close is None or nifty_prev_close <= 0:
            skipped.append((d, 'NoPrevClose')); continue

        gap = (nifty_open - nifty_prev_close) / nifty_prev_close

        # ── Compute signals ────────────────────────────────────────────────
        sigs = sig_map(gap, pind_ret, sp500_ret, sgx_ret, dax_ret, vix_ret)
        fired = [c for c in bear_combos if combo_fires(c, sigs)]

        if not fired:
            signal_days[d] = None; continue

        winning_combo = fired[0]

        # ── NIFTY 9:25 for strike ──────────────────────────────────────────
        row_925 = spot[spot['time'] == dtime(9, 25)]
        if row_925.empty:
            signal_days[d] = None; continue
        nifty_925 = float(row_925.iloc[0]['open'])
        atm    = round(nifty_925 / STRIKE_STEP) * STRIKE_STEP
        strike = atm - STRIKE_STEP   # 1-OTM put

        # ── Expiry ─────────────────────────────────────────────────────────
        exp = nearest_expiry(d)
        dte = (exp - d).days
        exp_str = exp.strftime('%d%b%y').upper()

        # ── Load option file ───────────────────────────────────────────────
        month_dir = DATA_2024 / expiry_folder(exp)
        opt_file  = month_dir / f"NIFTY-{exp_str}-{d.strftime('%d%b%y').upper()}.csv"
        if not opt_file.exists():
            signal_days[d] = None; continue

        try:
            opt = pd.read_csv(opt_file)
            opt.columns = [c.strip().lower() for c in opt.columns]
            opt['datetime'] = pd.to_datetime(opt['datetime'])
            opt['time']     = opt['datetime'].dt.time
            pe = opt[(opt['strike_price'] == strike) & (opt['right'].str.strip().str.upper() == 'PE')]
            if pe.empty:
                signal_days[d] = None; continue

            # Entry candle at 9:25
            entry_row = pe[pe['time'] == dtime(9, 25)]
            if entry_row.empty:
                entry_row = pe[pe['time'] >= dtime(9, 25)].head(1)
            if entry_row.empty:
                signal_days[d] = None; continue
            entry_price = float(entry_row.iloc[0]['open'])
            if entry_price <= 0:
                signal_days[d] = None; continue

            # Candles from 9:25 to 11:15
            candles = pe[(pe['time'] >= dtime(9, 25)) & (pe['time'] <= dtime(11, 15))].copy()
            candles = candles.reset_index(drop=True)

            signal_days[d] = {
                'entry_price':  entry_price,
                'candles':      candles,
                'dte':          dte,
                'strike':       strike,
                'combo':        winning_combo,
                'nifty_open':   nifty_open,
                'nifty_925':    nifty_925,
            }
        except Exception as e:
            print(f"  [warn] {d}: {e}")
            signal_days[d] = None

    # ── Save cache ─────────────────────────────────────────────────────────
    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(signal_days, f)

    valid = sum(1 for v in signal_days.values() if v is not None)
    print(f"Built trade cache: {len(signal_days)} dates, {valid} with valid signal+data.")
    print(f"  Skipped {len(skipped)} dates: "
          + ', '.join(f"{r}={sum(1 for _,x in skipped if x==r)}"
                      for r in dict.fromkeys(x for _,x in skipped)))
    print(f"  Cache saved: {CACHE_FILE}")


Fetching global market data ... done. 280 rows.
Loaded 10 bearish combos from v2_reliable_signals.csv
   1. Gap Up + Prev India DOWN + US UP + SGX UP
   2. Gap Up + Prev India DOWN + SGX UP + DAX UP
   3. Gap Up + Prev India DOWN + SGX UP + VIX Falling
   4. Gap Up + Prev India DOWN + SGX UP
   5. Gap Up + Prev India DOWN + US UP + DAX UP
   6. Gap Up + SGX UP + DAX UP + VIX Falling
   7. Gap Up + DAX UP + VIX Falling
   8. Gap Up + SGX UP + VIX Falling
   9. Gap Up + SGX UP + DAX UP
  10. Gap Up + US UP + SGX UP + DAX UP
Spot data loaded for 245 days.
Built trade cache: 189 dates, 56 with valid signal+data.
  Skipped 73 dates: Monday=53, NoSpotData=5, EventDay=7, Holiday=8
  Cache saved: c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v4\v4.1\results\trade_cache_v4_1.pkl


In [29]:
# ── Parameters dict passed to engine ──────────────────────────────────────────
params = dict(
    SL_PCT          = SL_PCT,
    TP_PCT          = TP_PCT,
    BASE_LOTS       = BASE_LOTS,
    MAX_LOTS        = MAX_LOTS,
    DTE0_LOTS       = DTE0_LOTS,
    LOT_SIZE        = LOT_SIZE,
    BEAR_N          = BEAR_N,
    ENTRY_TIME      = ENTRY_TIME,
    EXIT_TIME       = EXIT_TIME,
    INITIAL_CAPITAL = INITIAL_CAPITAL,
)

df_log = simulate_trades(signal_days, params)
print(f"Simulation complete: {len(df_log)} trades executed.")
df_log


Simulation complete: 56 trades executed.


,trade_num,date,strike,dte,entry,lots,sl_price,tp_price,exit_price,exit_reason,exit_time,pnl_pts,pnl_rs,charges_rs,capital_before,capital_after,drawdown_pct,combo
0,1,2024-01-09,21600,2,55.40,2,47.0900,77.56,77.5600,Target Hit,09:44:00,22.1600,3256.7829,67.2171,8310.0000,11566.7829,0.0000,Gap Up + Prev India DOWN + US UP + DAX UP
1,2,2024-01-11,21650,0,18.95,5,16.1075,26.53,16.1075,Stop Loss,09:26:00,-2.8425,-1125.3633,59.4258,11566.7829,10441.4196,9.7293,Gap Up + SGX UP + DAX UP + VIX Falling
2,3,2024-01-12,21700,6,105.05,1,89.2925,147.07,89.2925,Stop Loss,10:04:00,-15.7575,-1242.5673,60.7548,10441.4196,9198.8523,20.4718,Gap Up + SGX UP + VIX Falling
3,4,2024-01-19,21600,6,124.50,2,105.8250,174.30,105.8250,Stop Loss,10:31:00,-18.6750,-2880.5789,79.3289,18675.0000,15794.4211,15.4248,Gap Up + Prev India DOWN + US UP + DAX UP
4,5,2024-01-23,21650,2,61.70,3,52.4450,86.38,86.3800,Target Hit,10:00:00,24.6800,5472.3599,80.6401,15794.4211,21266.7810,0.0000,Gap Up + SGX UP + DAX UP + VIX Falling
5,6,2024-02-06,21750,2,129.85,2,110.3725,181.79,110.3725,Stop Loss,09:38:00,-19.4775,-3002.3345,80.7095,21266.7810,18264.4465,14.1175,Gap Up + Prev India DOWN + SGX UP + VIX Falling
6,7,2024-02-07,21950,1,67.45,3,57.3325,94.43,94.4300,Target Hit,09:45:00,26.9800,5986.7436,83.7564,18264.4465,24251.1901,0.0000,Gap Up + DAX UP + VIX Falling
7,8,2024-02-15,21850,0,36.65,5,31.1525,51.31,51.3100,Target Hit,09:26:00,14.6600,5417.1942,80.3058,24251.1901,29668.3843,0.0000,Gap Up + DAX UP + VIX Falling
8,9,2024-02-16,21950,6,127.95,3,108.7575,179.13,108.7575,Stop Loss,10:30:00,-19.1925,-4415.0413,96.7288,29668.3843,25253.3430,14.8813,Gap Up + SGX UP + DAX UP + VIX Falling
9,10,2024-02-23,22200,6,131.35,2,111.6475,183.89,111.6475,Stop Loss,10:17:00,-19.7025,-3036.4716,81.0966,25253.3430,22216.8714,25.1160,Gap Up + SGX UP + DAX UP + VIX Falling


In [30]:
metrics = compute_metrics(df_log, params)
print_summary(df_log, metrics, params)

version_meta = dict(
    version     = VERSION,
    key_change  = KEY_CHANGE,
    data_source = DATA_SRC,
)
out = save_results(metrics, params, version_meta, HERE / 'results')
print(f"\nRun ID: {out.stem}")


  BACKTEST RESULTS  ·  2024-01-09 to 2024-12-05
  Trades          : 56
  TP hit          : 32.1%  (18 trades)
  SL hit          : 60.7%  (34 trades)
  Time exit       : 7.1%  (4 trades)
  Profitable      : 35.7%
  Avg win (Rs)    :      6,348
  Avg loss (Rs)   :     -2,906
  Total P&L (Rs)  :     22,335
  Net return      : +382.8%
  XIRR            : +148.8%
  Max drawdown    : 47.4%
  Starting cap    : Rs      8,310
  Ending cap      : Rs     40,121
  Breakeven WR    : 27.3%  (SL=15% / TP=40%)

  Month     Trades   Wins    Win%      P&L (Rs)
  --------  ------  -----  ------  ------------
  2024-01        5      2   40.0%         3,481
  2024-02        5      2   40.0%           950
  2024-03        4      1   25.0%           578
  2024-04        6      2   33.3%        -2,924
  2024-05        4      2   50.0%         7,370
  2024-06        6      2   33.3%        -3,983
  2024-07        6      1   16.7%        -4,304
  2024-08        6      3   50.0%         6,209
  2024-09        4 